In [35]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [36]:
RAW_DIR = "../data/wisdm-dataset/raw"
SUBJECTS = range(1600, 1651)  # 51 subjects
SENSORS = [("phone", "accel"), ("phone", "gyro"), ("watch", "accel"), ("watch", "gyro")]

In [37]:
def get_sensor_data(filepath):
    df = pd.read_csv(
        filepath,
        header=None,
        names=["subject_id", "activity", "timeStamp", "x", "y", "z"],
    )

    df.iloc[:, -1] = df.iloc[:, -1].str.rstrip(";")

    df = df.astype({"z": float})

    df["timeStamp"] = pd.to_datetime(df["timeStamp"], unit="ns")
    df.drop(columns="subject_id", inplace=True)

    return df

In [166]:
def get_subject_data(subject_id):

    dfs: list[pd.DataFrame] = []

    for device, sensor in SENSORS:

        df = get_sensor_data(
            f"{RAW_DIR}/{device}/{sensor}/data_{subject_id}_{sensor}_{device}.txt"
        )

        df["device"] = device

        df = df.rename(
            columns={
                "x": f"x_{sensor}",
                "y": f"y_{sensor}",
                "z": f"z_{sensor}",
            }
        )

        dfs.append(df)

    for df in dfs:
        df.sort_values(by="timeStamp", inplace=True)

    merged_phone = pd.merge_asof(
        dfs[0] if dfs[0].shape[0] < dfs[1].shape[0] else dfs[1],
        dfs[0] if dfs[0].shape[0] >= dfs[1].shape[0] else dfs[1],
        on="timeStamp",
        by=["activity", "device"],
        direction="nearest",
        tolerance=pd.Timedelta("500ms"),
    )

    print(f"{merged_phone.shape[0]} rows of phone data")

    # return merged_phone

    merged_watch = pd.merge_asof(
        dfs[2] if dfs[2].shape[0] < dfs[3].shape[0] else dfs[3],
        dfs[2] if dfs[2].shape[0] >= dfs[3].shape[0] else dfs[3],
        on="timeStamp",
        by=["activity", "device"],
        direction="nearest",
        tolerance=pd.Timedelta("500ms"),
    )

    print(f"{merged_watch.shape[0]} rows of phone data")

    merged = pd.concat([merged_phone, merged_watch])

    merged = merged.iloc[:, [5, 0, 1, 2, 3, 4, 6, 7, 8]]

    merged["activity"] = merged["activity"].astype("category")
    merged["device"] = merged["device"].astype("category")

    return merged.sort_values(by=["device", "activity", "timeStamp"])

In [168]:
df = get_subject_data(1600)

df[df["activity"] == "B"].head(10)

64247 rows of phone data
65435 rows of phone data


,device,activity,timeStamp,x_gyro,y_gyro,z_gyro,x_accel,y_accel,z_accel
35694,phone,B,1970-01-03 21:59:47.770883934,0.410278,1.710571,0.504745,2.038742,3.077148,-1.053726
35695,phone,B,1970-01-03 21:59:47.821237937,-0.256134,0.380859,-0.295776,-2.558472,-2.738678,-2.098511
35696,phone,B,1970-01-03 21:59:47.871591941,-0.691132,-0.299866,0.030258,-1.355301,0.388412,-0.659851
35697,phone,B,1970-01-03 21:59:47.921945945,0.525040,-0.006882,0.728622,2.015015,3.416168,-7.555878
35698,phone,B,1970-01-03 21:59:47.972299949,-1.007584,-0.192322,-1.147400,-6.288239,14.711182,-3.468720
35699,phone,B,1970-01-03 21:59:48.022653953,-1.067123,-1.344116,-0.037460,-2.574417,15.563843,-5.310288
35700,phone,B,1970-01-03 21:59:48.073007957,-0.794479,-0.642075,-0.035736,5.656921,16.769974,-0.098587
35701,phone,B,1970-01-03 21:59:48.123361961,-0.337906,-0.599823,-0.319687,0.906815,9.314545,1.045059
35702,phone,B,1970-01-03 21:59:48.173715965,0.320038,-0.226364,0.440628,-1.792328,-2.573898,0.246811
35703,phone,B,1970-01-03 21:59:48.224069969,-0.234009,-0.399872,0.533295,-1.257858,-4.749680,-3.409119
